# 01 · Bronze incremental (requisitos 04 e 05)
Auto Loader lê os arquivos JSON Lines das pastas do Volume e grava **uma tabela Bronze por fonte**,
cópia fiel do que chegou + `arquivo_origem` e `carregado_em`.

**Roteiro das contagens (vão para o relatório):**
1. Envie o **lote 1** ao Volume → rode este notebook com `MOMENTO = "1 - apos lote 1"`
2. Envie o **lote 2** → rode com `MOMENTO = "2 - apos lote 2"` (só os arquivos novos entram)
3. Sem enviar nada → rode com `MOMENTO = "3 - reexecucao sem arquivo novo"` (deve acrescentar **zero** linhas)

In [0]:
MOMENTO = "3 - reexecucao sem arquivo novo"

BASE = "/Volumes/workspace/gs/chegada"
FONTES = ["clima", "ibge", "relatos"]

In [0]:
# Listagem do Volume (print para o requisito 04)
for fonte in FONTES:
    arquivos = sorted(f.name for f in dbutils.fs.ls(f"{BASE}/{fonte}"))
    print(f"{fonte}/: {len(arquivos)} arquivos -> {arquivos[0]} ... {arquivos[-1]}" if arquivos else f"{fonte}/: vazio")

clima/: 36 arquivos -> clima_2026-10-01.json ... clima_2026-11-05.json
ibge/: 36 arquivos -> ibge_2026-10-01.json ... ibge_2026-11-05.json
relatos/: 36 arquivos -> relatos_2026-10-01.json ... relatos_2026-11-05.json


In [0]:
from pyspark.sql import functions as F


def carregar_bronze(fonte: str) -> None:
    """Auto Loader: o checkpoint guarda quais arquivos já foram lidos, então só arquivo novo entra."""
    (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")                       # JSON Lines: um objeto por linha
        .option("cloudFiles.schemaLocation", f"{BASE}/_controle/schema_{fonte}")
        .load(f"{BASE}/{fonte}")
        .withColumn("arquivo_origem", F.col("_metadata.file_path"))
        .withColumn("carregado_em", F.current_timestamp())
        .writeStream
        .option("checkpointLocation", f"{BASE}/_controle/checkpoint_{fonte}")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"workspace.gs.bronze_{fonte}")
    ).awaitTermination()


for fonte in FONTES:
    carregar_bronze(fonte)
    print(f"bronze_{fonte} carregada")

bronze_clima carregada
bronze_ibge carregada
bronze_relatos carregada


In [0]:
# Registra as contagens deste momento numa tabela de controle (facilita o print comparativo)
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gs.controle_contagens_bronze
(momento STRING, tabela STRING, arquivos BIGINT, linhas BIGINT, medido_em TIMESTAMP)
""")
for fonte in FONTES:
    spark.sql(f"""
        INSERT INTO workspace.gs.controle_contagens_bronze
        SELECT '{MOMENTO}', 'bronze_{fonte}', COUNT(DISTINCT arquivo_origem), COUNT(*), current_timestamp()
        FROM workspace.gs.bronze_{fonte}
    """)

In [0]:
%sql

SELECT momento, tabela, arquivos, linhas, medido_em
FROM workspace.gs.controle_contagens_bronze
ORDER BY tabela, medido_em;

momento,tabela,arquivos,linhas,medido_em
1 - apos lote 1,bronze_clima,20,200,2026-11-05T21:18:06.640Z
2 - apos lote 2,bronze_clima,36,360,2026-11-05T21:23:47.343Z
3 - reexecucao sem arquivo novo,bronze_clima,36,360,2026-11-05T21:25:59.714Z
1 - apos lote 1,bronze_ibge,20,200,2026-11-05T21:18:09.333Z
2 - apos lote 2,bronze_ibge,36,360,2026-11-05T21:23:49.188Z
3 - reexecucao sem arquivo novo,bronze_ibge,36,360,2026-11-05T21:26:01.391Z
1 - apos lote 1,bronze_relatos,20,2938,2026-11-05T21:18:11.311Z
2 - apos lote 2,bronze_relatos,36,5531,2026-11-05T21:23:51.110Z
3 - reexecucao sem arquivo novo,bronze_relatos,36,5531,2026-11-05T21:26:03.285Z


In [0]:
%sql
-- Contagem por arquivo de origem (mostra que cada arquivo entrou uma única vez)
SELECT arquivo_origem, COUNT(*) AS linhas, MIN(carregado_em) AS carregado_em
FROM workspace.gs.bronze_relatos
GROUP BY arquivo_origem ORDER BY arquivo_origem;

arquivo_origem,linhas,carregado_em
/Volumes/workspace/gs/chegada/relatos/relatos_2026-10-01.json,168,2026-11-05T21:17:58.577Z
/Volumes/workspace/gs/chegada/relatos/relatos_2026-10-02.json,15,2026-11-05T21:17:58.577Z
/Volumes/workspace/gs/chegada/relatos/relatos_2026-10-03.json,155,2026-11-05T21:17:58.577Z
/Volumes/workspace/gs/chegada/relatos/relatos_2026-10-04.json,268,2026-11-05T21:17:58.577Z
/Volumes/workspace/gs/chegada/relatos/relatos_2026-10-05.json,134,2026-11-05T21:17:58.577Z
/Volumes/workspace/gs/chegada/relatos/relatos_2026-10-06.json,285,2026-11-05T21:17:58.577Z
/Volumes/workspace/gs/chegada/relatos/relatos_2026-10-07.json,123,2026-11-05T21:17:58.577Z
/Volumes/workspace/gs/chegada/relatos/relatos_2026-10-08.json,13,2026-11-05T21:17:58.577Z
/Volumes/workspace/gs/chegada/relatos/relatos_2026-10-09.json,260,2026-11-05T21:17:58.577Z
/Volumes/workspace/gs/chegada/relatos/relatos_2026-10-10.json,137,2026-11-05T21:17:58.577Z


In [0]:
%sql
-- Histórico da tabela: cada execução vira uma versão; a 3ª execução não deve ter linhas novas
DESCRIBE HISTORY workspace.gs.bronze_clima;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-11-05T21:23:30.000Z,76105308753725,arthurbaldisseracm@gmail.com,STREAMING UPDATE,"Map(queryId -> 877f913e-a90c-461c-9980-d6577d6ad1de, outputMode -> Append, clusterBy -> [], statsOnLoad -> true, epochId -> 2, clusteringOnWriteStatus -> Reason for skipping: Not attempted due to eligibility heuristic)",null,List(2594591947663599),90d3b4f4-1286-44e4-ad5b-854222ec94a9,1105-210208-szrvtpxj-v2n,2,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 160, numOutputBytes -> 36785, numAddedFiles -> 1)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-05T21:17:38.000Z,76105308753725,arthurbaldisseracm@gmail.com,STREAMING UPDATE,"Map(queryId -> 877f913e-a90c-461c-9980-d6577d6ad1de, outputMode -> Append, clusterBy -> [], statsOnLoad -> true, epochId -> 1, clusteringOnWriteStatus -> Reason for skipping: Not attempted due to eligibility heuristic)",null,List(2594591947663599),cdd0e0e8-e9a9-43bb-8826-90152c550bd2,1105-210208-szrvtpxj-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 78, numOutputRows -> 100, numOutputBytes -> 28626, numAddedFiles -> 1)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-05T21:17:36.000Z,76105308753725,arthurbaldisseracm@gmail.com,STREAMING UPDATE,"Map(queryId -> 877f913e-a90c-461c-9980-d6577d6ad1de, outputMode -> Append, clusterBy -> [], statsOnLoad -> true, epochId -> 0, clusteringOnWriteStatus -> Reason for skipping: Not attempted due to eligibility heuristic)",null,List(2594591947663599),cdd0e0e8-e9a9-43bb-8826-90152c550bd2,1105-210208-szrvtpxj-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 100, numOutputBytes -> 28730, numAddedFiles -> 1)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-05T21:17:25.000Z,76105308753725,arthurbaldisseracm@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""f6f7895b-8f2f-433f-8bce-91b23c561387"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-a9061865-c92c-4d70-a849-673b1d6ce9a9"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-8198fffc-d558-4fc9-8d66-f618470ae73f""}, statsOnLoad -> false, clusterByAuto -> true)",null,List(2594591947663599),cdd0e0e8-e9a9-43bb-8826-90152c550bd2,1105-210208-szrvtpxj-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


### Recomeçar do zero (só se precisar)
Apagar a tabela **e** o checkpoint juntos. Apagar só a tabela faz o Auto Loader achar que já leu tudo.
```python
for fonte in FONTES:
    spark.sql(f"DROP TABLE IF EXISTS workspace.gs.bronze_{fonte}")
    dbutils.fs.rm(f"{BASE}/_controle/checkpoint_{fonte}", True)
    dbutils.fs.rm(f"{BASE}/_controle/schema_{fonte}", True)
spark.sql("DROP TABLE IF EXISTS workspace.gs.controle_contagens_bronze")
```